# Embeddings Generation

Generate BiomedCLIP embeddings for medical images and text.

In [1]:
import torch
import open_clip
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import pandas as pd

c:\Users\Harsh\Desktop\major-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## BiomedCLIP Embedder Class

In [2]:
class BiomedCLIPEmbedder:
    def __init__(self, model_name='hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'):
        print(f"Loading {model_name}...")
        self.model, self.preprocess = open_clip.create_model_from_pretrained(model_name)
        self.tokenizer = open_clip.get_tokenizer(model_name)
        self.model.eval()
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)
        print(f"Model loaded on {self.device}")
        
    def encode_image(self, image_path):
        """Generate embedding for a single image."""
        img = Image.open(image_path).convert('RGB')
        img_tensor = self.preprocess(img).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            emb = self.model.encode_image(img_tensor)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        
        return emb.cpu()
    
    def encode_text(self, text):
        """Generate embedding for text."""
        tokens = self.tokenizer([text]).to(self.device)
        
        with torch.no_grad():
            emb = self.model.encode_text(tokens)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        
        return emb.cpu()
    
    def encode_dataset(self, df, image_col='image_path', output_path='embeddings.pt'):
        """Encode entire dataset."""
        embeddings = []
        metadata = []
        
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Encoding images"):
            try:
                emb = self.encode_image(row[image_col])
                embeddings.append(emb)
                metadata.append({k: v for k, v in row.items()})
            except Exception as e:
                print(f"Failed on {row[image_col]}: {e}")
        
        embeddings = torch.cat(embeddings, dim=0)
        
        torch.save({
            'embeddings': embeddings,
            'metadata': metadata
        }, output_path)
        
        print(f"✅ Saved {len(embeddings)} embeddings to {output_path}")
        return embeddings, metadata

## Initialize Embedder

In [3]:
embedder = BiomedCLIPEmbedder()

Loading hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Model loaded on cuda


## Test Single Image Encoding

In [4]:
# Test with a single image
# test_image = 'path/to/test/image.jpg'
# embedding = embedder.encode_image(test_image)
# print(f"Embedding shape: {embedding.shape}")

## Generate MIMIC Embeddings (if needed)

In [5]:
# Uncomment to generate MIMIC embeddings
# df_mimic = pd.read_csv('image-data/processed/mimic_master.csv')
# embedder.encode_dataset(df_mimic, output_path='embeddings/mimic_image_embeddings.pt')

## Generate IU-Xray Embeddings (if needed)

In [6]:
# Uncomment to generate IU-Xray embeddings
# df_iu = pd.read_csv('iu_xray.csv')
# embedder.encode_dataset(df_iu, output_path='embeddings/iu_biomedclip_embeddings.pt')

## Load Existing Embeddings

In [ ]:
# Load pre-computed embeddings
data = torch.load('embeddings/mimic_image_embeddings.pt')
embeddings = data['embeddings']
metadata = data['metadata']

print(f"Loaded {len(embeddings)} embeddings")
print(f"Embedding dimension: {embeddings.shape[1]}")

C:\Users\Harsh\AppData\Local\Temp\ipykernel_56344\665958857.py:3: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\python_variable_indexing.cpp:351.)
  embeddings = data['embeddings']


IndexError: too many indices for tensor of dimension 2